Đọc dữ liệu

In [1]:

import os
import numpy as np
import pandas as pd

# Đường dẫn dữ liệu
file_path = os.path.join(
    "..", "..", "data", "raw", "Diamonds Prices2022.csv"
)

# Đọc dữ liệu
diamonds = pd.read_csv(file_path, sep=',')

print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Thông tin dữ liệu
diamonds.info()

print("\n--- 5 dòng đầu tiên ---")
print(diamonds.head())

print("\n--- Giá trị phân loại ---")
print(f"Cut: {diamonds['cut'].unique()}")
print(f"Color: {diamonds['color'].unique()}")
print(f"Clarity: {diamonds['clarity'].unique()}")

print("\n--- Thống kê mô tả ---")
print(diamonds.describe())


Kích thước tập dữ liệu ban đầu: (53943, 11)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53943 entries, 0 to 53942
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53943 non-null  int64  
 1   carat       53943 non-null  float64
 2   cut         53943 non-null  object 
 3   color       53943 non-null  object 
 4   clarity     53943 non-null  object 
 5   depth       53943 non-null  float64
 6   table       53943 non-null  float64
 7   price       53943 non-null  int64  
 8   x           53943 non-null  float64
 9   y           53943 non-null  float64
 10  z           53943 non-null  float64
dtypes: float64(6), int64(2), object(3)
memory usage: 4.5+ MB

--- 5 dòng đầu tiên ---
   Unnamed: 0  carat      cut color clarity  depth  table  price     x     y  \
0           1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98   
1           2   0.21  Premium     E     SI1   59.8   61.0    326  

Tiền xử lý dữ liệu

In [2]:

from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ cột không mang thông tin
if 'Unnamed: 0' in diamonds.columns:
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ các dòng có kích thước không hợp lệ
initial_rows = len(diamonds)

diamonds = diamonds[
    (diamonds['x'] > 0) &
    (diamonds['y'] > 0) &
    (diamonds['z'] > 0)
].copy()

invalid_dimensions = initial_rows - len(diamonds)

print(
    f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu "
    "(x, y, z <= 0)."
)

# 3: Loại bỏ các dòng trùng lặp
num_duplicates = diamonds.duplicated().sum()

diamonds = diamonds.drop_duplicates()

print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa Ordinal
cut_categories = [
    'Fair', 'Good', 'Very Good', 'Premium', 'Ideal'
]

color_categories = [
    'J', 'I', 'H', 'G', 'F', 'E', 'D'
]

clarity_categories = [
    'I1', 'SI2', 'SI1', 'VS2',
    'VS1', 'VVS2', 'VVS1', 'IF'
]

encoder = OrdinalEncoder(
    categories=[
        cut_categories,
        color_categories,
        clarity_categories
    ]
)

diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(
    diamonds[['cut', 'color', 'clarity']]
)

# 5: Kiểm tra dữ liệu sau tiền xử lý
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

print("--- 5 dòng đầu sau tiền xử lý ---")
print(diamonds.head())



Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
0   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
1   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
2   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
3   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
4   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


Chia tập dữ liệu

In [3]:

from sklearn.model_selection import train_test_split

# X là các thuộc tính
X = diamonds.drop(columns=["price"])

# y là nhãn giá kim cương
y = diamonds["price"]

# Chia dữ liệu Train/Test theo tỷ lệ 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Chia thêm tập Validation từ Train
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)

print(
    f"Có {X_train.shape[0]} mẫu huấn luyện "
    f"và {X_test.shape[0]} mẫu kiểm tra"
)

print(f"Train dùng để tìm tham số: {X_train_sub.shape[0]} mẫu")
print(f"Validation: {X_val.shape[0]} mẫu")


Có 43020 mẫu huấn luyện và 10755 mẫu kiểm tra
Train dùng để tìm tham số: 34416 mẫu
Validation: 8604 mẫu


Training và tìm bộ tham số tốt nhất

In [4]:

# Import mô hình Random Forest hồi quy
from sklearn.ensemble import RandomForestRegressor

# Import các hàm đánh giá
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Import thư viện đo thời gian
import time

# Khai báo các giá trị tham số cần thử
n_estimators_values = [50, 100]
max_depth_values = [10, 15]
min_samples_leaf_values = [1, 2]
min_samples_split_values = [2, 5]

# Khởi tạo biến lưu kết quả tốt nhất
best_rmse = float('inf')
best_params = None
best_model = None
best_times = None

# Đếm số tổ hợp đã chạy
count = 0

# Bắt đầu tìm kiếm tham số
for n_estimators in n_estimators_values:
    for max_depth in max_depth_values:
        for min_samples_leaf in min_samples_leaf_values:
            for min_samples_split in min_samples_split_values:

                count += 1

                # Khởi tạo mô hình với bộ tham số hiện tại
                model = RandomForestRegressor(
                    criterion="squared_error",
                    random_state=42,
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    min_samples_leaf=min_samples_leaf,
                    min_samples_split=min_samples_split
                )

                # Đo thời gian huấn luyện
                start_time = time.perf_counter()

                # Huấn luyện trên tập Train phụ
                model.fit(X_train_sub, y_train_sub)

                end_time = time.perf_counter()

                # Dự đoán trên tập Validation
                y_pred_val = model.predict(X_val)

                # Tính RMSE Validation
                rmse = np.sqrt(
                    mean_squared_error(y_val, y_pred_val)
                )

                # Cập nhật bộ tham số tốt nhất
                if rmse < best_rmse:
                    best_rmse = rmse

                    best_params = {
                        'n_estimators': n_estimators,
                        'max_depth': max_depth,
                        'min_samples_leaf': min_samples_leaf,
                        'min_samples_split': min_samples_split
                    }

                    best_times = end_time - start_time
                    best_model = model

                print(
                    f"Đã chạy {count}/16 tổ hợp - "
                    f"RMSE Validation: {rmse:.4f}"
                )

# In bộ tham số tốt nhất
print("\nTham số tốt nhất:", best_params)
print(f"RMSE Validation tốt nhất: {best_rmse:.4f}")
print(f"Thời gian train khi tìm được bộ tốt nhất: {best_times:.4f} giây")

# Huấn luyện lại mô hình trên toàn bộ tập Train
best_model = RandomForestRegressor(
    criterion="squared_error",
    random_state=42,
    **best_params
)

start_time = time.perf_counter()

best_model.fit(X_train, y_train)

training_time = time.perf_counter() - start_time

# Dự đoán trên Train và Test
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

# Đánh giá trên Train
mae_train = mean_absolute_error(y_train, y_pred_train)
mse_train = mean_squared_error(y_train, y_pred_train)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_pred_train)

# Đánh giá trên Test
mae_test = mean_absolute_error(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, y_pred_test)

# In kết quả
print("\nKẾT QUẢ RANDOM FOREST - HOLD OUT MỨC 3")

print(f"\nMAE Train: {mae_train:.4f}")
print(f"MSE Train: {mse_train:.4f}")
print(f"RMSE Train: {rmse_train:.4f}")
print(f"R² Train: {r2_train:.4f}")

print(f"\nMAE Test: {mae_test:.4f}")
print(f"MSE Test: {mse_test:.4f}")
print(f"RMSE Test: {rmse_test:.4f}")
print(f"R² Test: {r2_test:.4f}")

print(f"\nThời gian Train mô hình cuối: {training_time:.4f} giây")


Đã chạy 1/16 tổ hợp - RMSE Validation: 584.9575
Đã chạy 2/16 tổ hợp - RMSE Validation: 586.3465
Đã chạy 3/16 tổ hợp - RMSE Validation: 585.1744
Đã chạy 4/16 tổ hợp - RMSE Validation: 584.7969
Đã chạy 5/16 tổ hợp - RMSE Validation: 568.6803
Đã chạy 6/16 tổ hợp - RMSE Validation: 567.3981
Đã chạy 7/16 tổ hợp - RMSE Validation: 565.5079
Đã chạy 8/16 tổ hợp - RMSE Validation: 566.4829
Đã chạy 9/16 tổ hợp - RMSE Validation: 582.0631
Đã chạy 10/16 tổ hợp - RMSE Validation: 582.3764
Đã chạy 11/16 tổ hợp - RMSE Validation: 582.7200
Đã chạy 12/16 tổ hợp - RMSE Validation: 581.9297
Đã chạy 13/16 tổ hợp - RMSE Validation: 565.6290
Đã chạy 14/16 tổ hợp - RMSE Validation: 566.2040
Đã chạy 15/16 tổ hợp - RMSE Validation: 565.0538
Đã chạy 16/16 tổ hợp - RMSE Validation: 565.5596

Tham số tốt nhất: {'n_estimators': 100, 'max_depth': 15, 'min_samples_leaf': 2, 'min_samples_split': 2}
RMSE Validation tốt nhất: 565.0538
Thời gian train khi tìm được bộ tốt nhất: 10.5154 giây

KẾT QUẢ RANDOM FOREST - HOLD 